In [17]:
import random
import math

In [37]:
# To keep track of more complex equation and derivatives

class Value:

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        r = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += r.grad
            other.grad += r.grad
        r._backward = _backward
        
        return r

    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        r = Value(self.data - other.data, (self, other), '-')

        def _backward():
            self.grad += r.grad
            other.grad += -r.grad
        r._backward = _backward
        
        return r

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        r = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += r.grad * other.data
            other.grad += r.grad * self.data
        r._backward = _backward
        
        return r

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        r = Value(self.data / other.data, (self, other), '/')
        
        def _backward():
            self.grad += r.grad / other.data
            other.grad += r.grad / self.data
        r._backward = _backward
        
        return r

    def tanh(self):
        x = self.data
        upper = math.exp(2*x) - 1 
        lower = math.exp(2*x) + 1
        t = upper / lower
        r = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * r.grad
        r._backward = _backward
        
        return r

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [38]:
# y = 3x + 2 + noise
data = []
for _ in range(50):
    x = random.uniform(-5, 5)
    y = 3*x + 2 + random.uniform(-0.5, 0.5)
    data.append((Value(x), y))
data[0]

(Value(data=4.746623548736007), 16.085641023780116)

In [39]:
# Random Parameters Initialisation
w = Value(random.uniform(-1.0, 1.0))
b = Value(random.uniform(-1.0, 1.0))

In [40]:
# learning rate
lr = 0.01
training_steps = 100

In [41]:

for step in range(training_steps):
    # forward
    loss = Value(0) # initial
    for x, y in data:
        pred = x * w + b # where did this linear equation came from ? How did we know to choose this ? We are assuming the true relationship is a line. The network does not "know" this; we chose a linear model because the data is linear. For curved data, we would add hidden layers and activation functions.
        diff = pred - y 
        loss = loss + diff * diff # why sqaure of this ? Why adding loss ? diff * diff makes every error positive (so negative and positive errors don't cancel) and punishes large errors more aggressively. The sum adds up the error across all data points.
    
    loss = loss / float(len(data)) # MEAN squared error
    
    # backward
    w.grad = 0.0
    b.grad = 0.0
    loss.backward()

    #update
    w.data -= lr * w.grad
    b.data -= lr * b.grad

    if step % 10 == 0:
        print(f"step {step}: loss={loss.data:.3f}, w={w.data:.3f}, b={b.data:.3f}")

print(f"Final: w={w.data:3f} (target 3.0), b={b.data:.3f} (target 2.0)")


step 0: loss=66.434, w=0.359, b=0.508
step 10: loss=4.809, w=2.398, b=0.795
step 20: loss=1.102, w=2.869, b=1.005
step 30: loss=0.648, w=2.976, b=1.170
step 40: loss=0.454, w=2.998, b=1.304
step 50: loss=0.331, w=3.002, b=1.413
step 60: loss=0.249, w=3.002, b=1.503
step 70: loss=0.194, w=3.000, b=1.575
step 80: loss=0.157, w=2.999, b=1.635
step 90: loss=0.132, w=2.998, b=1.684
Final: w=2.997531 (target 3.0), b=1.720 (target 2.0)
